# CARDIODIAG Kaggle Quickstart

Fast, leakage-safe comparison. This is not the full nested stability/reliability protocol.

In [ ]:
from pathlib import Path
import sys, json, hashlib, time, platform
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import average_precision_score, roc_auc_score, log_loss

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and Path("/kaggle/input").exists():
    candidates = list(Path("/kaggle/input").glob("**/src"))
    if candidates: PROJECT_ROOT = candidates[0].parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from cardiodiag.data import load_dataset, drop_identifier_columns, summarize_dataset
from cardiodiag.evaluation import evaluate_predictions
from cardiodiag.preprocessing import TrainingOnlyCleaner, make_tabular_preprocessor
from cardiodiag.provenance import collect_provenance
print("Python", platform.python_version(), "root", PROJECT_ROOT)
def find_dataset(filename):
    candidates = [PROJECT_ROOT / "data" / filename, PROJECT_ROOT / filename]
    if Path("/kaggle/input").exists(): candidates += list(Path("/kaggle/input").glob(f"**/{filename}"))
    for path in candidates:
        if path.exists(): return path
    return None

DATASETS = {
    "mendeley": (find_dataset("MendeleyCardiovascular_Disease_Dataset.csv"), "target"),
    "heart5": (find_dataset("heart5.csv"), "HeartDisease"),
    "kaggle70k": (find_dataset("cardio_train.csv"), "cardio"),
}
print({k: str(v[0]) if v[0] else "not attached" for k,v in DATASETS.items()})


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

def pipeline_for(X, model):
    categorical=list(X.attrs.get("categorical_columns", [])); numeric=list(X.attrs.get("numeric_columns", []))
    return Pipeline([("cleaner", TrainingOnlyCleaner(categorical_columns=tuple(categorical))), ("preprocessor", make_tabular_preprocessor(numeric, categorical)), ("model", model)])

def run_cv(path, target, seed=42, outer_splits=3, inner_splits=2):
    X,y=load_dataset(path,target); X,_=drop_identifier_columns(X)
    outer=StratifiedKFold(n_splits=outer_splits,shuffle=True,random_state=seed)
    specs={
      "logistic":(LogisticRegression(max_iter=1500,class_weight="balanced",random_state=seed),{"model__C":[0.1,1.0,10.0]}),
      "random_forest":(RandomForestClassifier(n_estimators=150,n_jobs=1,class_weight="balanced",random_state=seed),{"model__max_depth":[None,16]}),
      "hist_gradient_boosting":(HistGradientBoostingClassifier(random_state=seed),{"model__max_leaf_nodes":[15,31],"model__max_iter":[150]}),
    }
    records=[]; predictions={}
    for name,(model,grid) in specs.items():
        pred=np.full(len(y),np.nan); folds=[]
        for fold,(train,test) in enumerate(outer.split(X,y)):
            search=GridSearchCV(pipeline_for(X.iloc[train],model),grid,scoring="roc_auc",cv=inner_splits,n_jobs=1,refit=True)
            search.fit(X.iloc[train],y.iloc[train]); p=search.predict_proba(X.iloc[test])[:,1]; pred[test]=p
            folds.append({"fold":fold,"best_params":search.best_params_,"metrics":evaluate_predictions(y.iloc[test].to_numpy(),p)})
        predictions[name]=pred; records.append({"model":name,"pooled":evaluate_predictions(y.to_numpy(),pred),"folds":folds})
    ensemble=np.mean([predictions["logistic"],predictions["random_forest"],predictions["hist_gradient_boosting"]],axis=0)
    records.append({"model":"fixed_soft_voting","pooled":evaluate_predictions(y.to_numpy(),ensemble)})
    return {"dataset":str(path),"target":target,"records":records,"provenance":collect_provenance(path,target,"notebook_quick_cv",{"seed":seed,"outer_splits":outer_splits,"inner_splits":inner_splits})}

# Run only after selecting an attached dataset and reviewing its protocol.
# example_result = run_cv(DATASETS["mendeley"][0], DATASETS["mendeley"][1])


## Train compact baselines on an attached dataset

In [ ]:
selected = next(((p,t) for p,t in DATASETS.values() if p is not None), (None,None))
if selected[0] is None: raise FileNotFoundError("Attach Mendeley, heart5, or cardio_train.csv and rerun.")
result = run_cv(*selected, seed=42, outer_splits=3, inner_splits=2)
for record in result["records"]: print(record["model"], record["pooled"].get("roc_auc"), record["pooled"].get("pr_auc"))


## Save outputs and limitations

In [ ]:
out=PROJECT_ROOT/"results"/"quickstart"/"quickstart_metrics.json"
out.parent.mkdir(parents=True,exist_ok=True); out.write_text(json.dumps(result,indent=2,default=str),encoding="utf-8")
print(out)


The quickstart runs Logistic Regression, Random Forest, HistGradientBoosting, and fixed soft voting on identical folds. It does not replace the complete custom CARDIODIAG protocol.